# 데이터 탐색과 첫 분류 모델

원본은 수정하지 않습니다. DATA_DIR을 success/fail 폴더가 있는 위치로 설정하세요. 전체 라벨을 보는 탐색은 이후 특징 선택에 영향을 줄 수 있으므로 검증 결과를 과도하게 해석하지 않습니다.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise RuntimeError("레포 루트 또는 notebooks 폴더에서 실행하세요")
sys.path.insert(0, str(ROOT))
DATA_DIR = ROOT / "data"  # 필요하면 외부 데이터 경로로 변경
from src.data import load_dataset
from src.features import extract_features
from src.eda import plot_episodes
from src.train import evaluate
import pandas as pd
from IPython.display import display, Image

## 데이터 점검

검사 오류가 있으면 기본 로더가 중단합니다. 요약표는 모델 입력이 아닙니다.

In [ ]:
episodes = load_dataset(DATA_DIR)
summary = pd.DataFrame([e.summary for e in episodes])
display(summary.drop(columns="path"))
display(summary.groupby("class_name")["n_frames"].agg(["count", "min", "max", "sum"]))

## 관절 움직임 비교

상대 시간 0~1은 길이만 맞춥니다. 작업 단계가 같은 시점에 일어난다고 가정하지 않습니다. 얇은 선은 각 에피소드, 굵은 선은 클래스 평균입니다.

In [ ]:
plot_episodes(episodes, ROOT / "reports/eda")
display(Image(filename=str(ROOT / "reports/eda/state_by_class.png"), width=1000))
display(Image(filename=str(ROOT / "reports/eda/action_by_class.png"), width=1000))

## 특징 추출과 5겹 교차검증

52개 채널의 평균·표준편차·범위·끝-시작 차이로 208개 특징을 만듭니다. ID와 경로는 metadata에만 있고 정답은 y에 있습니다. 상수 제거와 표준화는 각 학습 fold에서만 fit합니다.

In [ ]:
X, y, metadata = extract_features(episodes)
print("X:", X.shape, "labels:", y.value_counts().to_dict())
metrics, predictions = evaluate(X, y, seed=42)
display(pd.Series({k: v for k, v in metrics.items() if isinstance(v, (int, float))}))
display(pd.DataFrame(metrics["confusion_matrix"], index=["true_fail", "true_success"], columns=["pred_fail", "pred_success"]))
display(pd.concat([metadata[["episode_id"]], predictions], axis=1))

## 해석

다수 클래스 기준 정확도와 함께 비교하세요. 48개 에피소드의 단일 교차검증은 독립 테스트 성능을 보장하지 않습니다. 이 단계에는 군집화·품질 점수·하이퍼파라미터 탐색은 포함하지 않습니다. 결과 파일 저장은 README의 python -m src.train 명령을 사용하세요.